# Scratch pad

<img src="illustrations/playground_cover.png" alt="A scratch pad beside the lecture notebook, for trying thread ideas by hand." width="100%">

A bench next to the lecture. The guided tour, one picture per demo, is `Day06_Live_Demos.ipynb`. The cells below check two libraries, then play with a downloader thread, a return value, and `join()`.


## Two libraries to install

<img src="illustrations/playground_imports.png" alt="The notebooks asks for requests and numpy. Both are missing in this environment." width="100%">

`requests` and `numpy` are the packages these two cells ask for. In this environment each import raises `ModuleNotFoundError`. Everything after them uses the standard library, so the thread experiments run as they are.


In [1]:
import requests

ModuleNotFoundError: No module named 'requests'

In [2]:
import numpy

ModuleNotFoundError: No module named 'numpy'

## A downloader, and a place to put the squares

<img src="illustrations/playground_mix.png" alt="Downloader reports size 500. Three slots keep the squares 9, 25, and 49." width="100%">

`Downloader` is a `Thread`. `start()` calls `run()` on the new thread, and after `join()` the size is sitting on the object: `len("ai-47") * 100` is **500**. Under that, `square(4)` computes 16 and `join()` returns `None`. `square_into` writes each square into its own list slot, and the results survive as `[9, 25, 49]`.


In [7]:
import threading, time

class Downloader(threading.Thread):
    def __init__(self, url):
        super().__init__(name=f"dl-{url}")
        self.url = url
        self.size = None

    def run(self):               # start() calls run() in the new thread
        time.sleep(0.1)
        self.size = len(self.url) * 100
        print(f"--- is running  {self.url} ---")

d = Downloader("ai-47")
d.start()
threading.active_count()
d.join()
print(d.name, d.size)
################################ dl-iti.gov.eg 1000

def square(n):
    return n * n

t = threading.Thread(target=square, args=[4])
t.start()
print("join returned:", t.join())                         # None: the 16 is lost

def square_into(n, results, i):
    time.sleep(0.1)
    results[i] = n * n           # each thread writes its own slot
#
numbers = [3, 5, 7]
results = [None] * len(numbers)
threads = [threading.Thread(target=square_into, args=(n, results, i)) for i, n in enumerate(numbers)]
for t in threads: t.start()
for t in threads: t.join()
print(results)

--- is running  ai-47 ---
dl-ai-47 500
join returned: None
[9, 25, 49]


## Three workers, then join

<img src="illustrations/playground_join.png" alt="Three workers finish, and only then does main ask all done?" width="100%">

Three workers sleep for half a second. The cell starts them and joins every one, so `main: all done?` prints once they have all finished. The three `worker: finished` lines can land on the same row, because `print` writes its characters as they come. That tangle is demo 8 in the lecture notebook.


In [6]:
import threading, time

USE_JOIN = True      # change to True and run again

def work():
    time.sleep(0.5)
    print("worker: finished")

threads = [threading.Thread(target=work) for i  in range(3)]
for t in threads: t.start()
for t in threads: t.join() 
print(threads)

print("main: all done?")

worker: finishedworker: finished

worker: finished
[<Thread(Thread-9 (work), stopped 135473603278528)>, <Thread(Thread-10 (work), stopped 135474153268928)>, <Thread(Thread-11 (work), stopped 135473594885824)>]
main: all done?
